# Figure 1 — ENCODE BPNet models: test-set performance and motif information content

Inputs (all built by `pipeline/`, see the README):
- `data/auroc_summary.parquet` — per-model test-set counts AUROC with / without the control track (step 01)
- `data/ic_motifs_data.parquet` — per-motif core IC and peak coverage of all 2,282 models (step 02)
- ENCODE files of the BHLHE40 K562 model under `$BPNET_DATA/PAPER/K562/BHLHE40/` (step 00)

No GPU needed.

In [ ]:
import sys
sys.path.insert(0, '../modules')        # run from notebooks/
import paths                            # data root, genome, repo locations (env BPNET_DATA, MISHA_HG38)
from plot_setup import fig_path         # editable PDF text; figures/ in the repository
%config InlineBackend.figure_format = 'retina'

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import moodspy as mp
from moodspy import plotting
import pymisha as pm
from scipy.stats import pearsonr, spearmanr, gaussian_kde
from pypalettes import load_cmap
from plotnine import *
from plotnine_extensions import *       # theme_cowplot, theme_pubclean, rasterize, geom_hex, geom_text_repel

pm.gsetroot(paths.MISHA_HG38)

## A. Test-set AUROC with the control track (`auroc_ctl`) and without it (`auroc_noctl`)
Mean over the 5 test folds, peaks vs GC-matched nonpeaks, one value per model.

In [ ]:
aurocs = pd.read_parquet(paths.DATA / 'auroc_summary.parquet')[['auroc_ctl', 'auroc_noctl']]

long = aurocs.unstack().reset_index().rename(columns={'level_0': 'cond', 'level_1': 'exp_id', 0: 'auroc'})

encode_auroc = (
    ggplot(long, aes(x='auroc', fill='cond'))
    + geom_density(alpha=.5)
    + scale_fill_cmap_d('Greens')
    + theme_cowplot()
)
encode_auroc.show()
encode_auroc.save(filename=fig_path('encode_qc.pdf'))

## B–C. A composite and a regular motif: BHLHE40 (K562) pattern_0 vs pattern_8
TF-MoDISco counts-head motifs (untrimmed PFMs), their total information content, and the fraction of peaks with a
Fi-NeMo hit (λ = 0.7) within ±500 bp of the summit.

In [ ]:
testcase_dir = paths.PAPER / 'K562' / 'BHLHE40'
SHOW = ['pattern_0', 'pattern_8']

[pfm] = glob.glob(f'{testcase_dir}/sequence_motifs__*/counts/*.meme/*.PFM.meme')
motifs = {m.id.rsplit('_counts_', 1)[1]: m for m in mp.read_motifs(pfm)}    # 'pattern_0', ...

In [ ]:
fig, ax = plt.subplots(2, 1, sharex=True, constrained_layout=True, figsize=(12, 6))
for axi, k in zip(ax, SHOW):
    plotting.plot_logo(motifs[k], ax=axi, kind='ICM')
    axi.set_ylim([0, 2])
ax[1].set_xticks(np.arange(0, 60, 10))
fig.savefig(fig_path('BHLHE40_seqlets.pdf'))

In [ ]:
fig, ax = plt.subplots(2, 1, sharex=True, sharey=True, constrained_layout=True, figsize=(1, 6))
for axi, k in zip(ax, SHOW):
    axi.bar('ic', mp.icm(motifs[k]).sum())
    axi.set_ylabel('sum(IC)')
fig.savefig(fig_path('BHLHE40_seqlets_ic_sum.pdf'))

In [ ]:
[ppath] = glob.glob(f'{testcase_dir}/training_and_test_regions__*/peaks.all_input_regions.*.bed.gz')
p = pd.read_csv(ppath, sep='\t', header=None)

W = 1000                                   # window width (bp)
summit = p[1] + p[9]                       # narrowPeak: col 1 = start, col 9 = summit offset from start
peaks_regions = pd.DataFrame({'chrom': p[0], 'start': summit - W // 2, 'end': summit + W // 2})
peaks_regions = peaks_regions[peaks_regions.start >= 0].reset_index(drop=True)   # drop windows off the chromosome start
peaks_regions['peak_idx'] = np.arange(len(peaks_regions))

[mpath] = glob.glob(f'{testcase_dir}/sequence_motifs_instances__*/counts/*lambda_0p7/*.bed')
motif_regions = pm.gintervals_import_bed(mpath)
motif_regions['hit_id'] = np.arange(len(motif_regions))
ov = pm.gintervals_neighbors(motif_regions, peaks_regions, mindist=0, maxdist=0, maxneighbors=10_000)
frac_peaks = ov.groupby('name').peak_idx.nunique() / len(peaks_regions)
frac_peaks.index = frac_peaks.index.str.rsplit('_counts_', n=1).str[1]
frac_peaks

In [ ]:
pnine_df = frac_peaks.loc[SHOW].rename('frac').rename_axis('name').reset_index().assign(x='frac')

p = (
    ggplot(pnine_df, aes(x='x', y='frac'))
    + geom_col()
    + facet_wrap('name', nrow=2, scales='fixed')
    + labs(x='')
    + scale_y_log10()
    + theme_classic()
    + theme(figure_size=(2, 6))
)
p.show()
p.save(filename=fig_path('BHLHE40_seqlets_frac_peaks.pdf'))

## D–G. Information content of the 20-bp motif core across all models
`core_ic` = bits in the central 20 bp of each untrimmed TF-MoDISco motif; `core_ic / 2` = effective bp
(2 bits per fully specified base). `frac_peaks_with_motif` = fraction of the model's peaks with ≥ 1 Fi-NeMo hit.

In [ ]:
summ = pd.read_parquet(paths.DATA / 'ic_motifs_data.parquet')
pattern_0 = summ[summ.motif_name.apply(lambda x: 'pos_pattern' in x and 'pattern_0' in x)]


def proc_short_name(name):
    """Motif groups by TF-MoDISco rank (patterns are numbered from 0; labels count from 1)."""
    p_num = int(name.split('_')[-1])
    if p_num <= 4:
        return 'pattern 1 to 5'
    elif p_num <= 9:
        return 'pattern 6 to 10'
    else:
        return 'pattern>10'


summ['sum_name'] = summ.short_name.apply(proc_short_name)
summ.sum_name.value_counts()

In [ ]:
fun = load_cmap('BluGrn')

p = (
    ggplot(summ.sort_values(by='frac_peaks_with_motif'), aes(x='core_ic/2', y='frac_peaks_with_motif', color='sum_name'))
    + rasterize(geom_hex(bins=40), dpi=150)
    + scale_fill_gradientn(colors=fun.hex, limits=[1, 40])
    + facet_wrap('sum_name')
    + theme_cowplot()
    + theme(figure_size=(18, 10))
    + scale_y_continuous(expand=(0, 0, 0.05, 0))
    + guides(fill=guide_colorbar(display='rectangles', nbin=100))
)
p.show()
p.save(filename=fig_path('ic_hex.pdf'))

In [ ]:
def wmean_ic(g):
    """Per model: mean effective bp of its motifs, weighted by the fraction of peaks each motif occurs in."""
    w = g.frac_peaks_with_motif
    return pd.Series({'Effective_bp': np.average(g.core_ic / 2, weights=w) if w.sum() > 0 else np.nan,
                      'n_motifs': len(g), 'sum_frac': w.sum()})


per_model = (summ.groupby(['tf', 'cell_line', 'TF_CellLine', 'experiment', 'bpnet_set', 'dir'])
                 .apply(wmean_ic, include_groups=False).reset_index())
per_tf = (per_model.groupby('tf')
                   .agg(Effective_bp=('Effective_bp', 'mean'), sd=('Effective_bp', 'std'),
                        n_models=('dir', 'size'), n_cell_lines=('cell_line', 'nunique'))
                   .reset_index())

In [ ]:
d = per_tf.dropna(subset=['Effective_bp']).sort_values('Effective_bp').reset_index(drop=True)
d['ecdf'] = np.arange(1, len(d) + 1) / len(d)

cm = 1 / 2.54
p = (
    ggplot()
    + geom_step(d, aes(x='Effective_bp', y='ecdf'), direction='hv')
    + labs(x='Effective_bp', y='ECDF')
    + theme_pubclean()
    + theme(figure_size=(12 * cm, 9 * cm))
)
p.show()
p.save(filename=fig_path('weighted_ic_mean.pdf'))

In [ ]:
df = d.sort_values('Effective_bp', ascending=False).reset_index(drop=True)
df['rank'] = np.arange(1, len(df) + 1)
top = df['rank'] <= 10
df['lab'] = np.where(top, df['tf'], '')

p = (
    ggplot(df, aes('rank', 'Effective_bp'))
    + geom_line(color='grey')
    + geom_point(df[top], aes('rank', 'Effective_bp'), color='teal', size=5, stroke=0)
    + geom_text_repel(
        aes(label='lab'),
        size=7,
        point_size=4,                    # match the geom_point size
        direction='y',                   # stack labels vertically...
        xlim=(len(df) * 0.08, None),     # ...in the empty space to the right of the head
        segment_color='grey',
        segment_size=0.3,
        max_time=3,
    )
    + labs(x='TF rank', y='Weighted mean (Effective bp)')
    + theme_cowplot()
    + theme(figure_size=(10, 5))
)
p.show()
p.save(filename=fig_path('ic_tf_ranked.pdf'))

In [ ]:
p0_mean = pattern_0.groupby('tf')['core_ic'].mean().reset_index()
d = df.merge(p0_mean, on='tf')[['Effective_bp', 'core_ic']].dropna()
r, pval = pearsonr(d.Effective_bp, d.core_ic)
rho, _ = spearmanr(d.Effective_bp, d.core_ic)
print(f'Pearson r = {r:.3f} (p = {pval:.1e}), Spearman rho = {rho:.3f}, n = {len(d)}')

xy = d[['Effective_bp', 'core_ic']].to_numpy().T
d = d.assign(density=gaussian_kde(xy)(xy)).sort_values('density')   # dense points drawn last

fun = load_cmap('Ernst', reverse=True)
ic_comp_p0 = (
    ggplot(d, aes(x='Effective_bp', y='core_ic/2', color='density'))
    + geom_point(size=3, alpha=.7, stroke=0)
    + scale_color_gradientn(colors=fun.hex, breaks=[d.density.min(), d.density.max()], labels=['low', 'high'])
    + geom_smooth(aes(group=1), method='lm', se=True, fullrange=True, color='black')
    + annotate('text', x=d.Effective_bp.min(), y=d.core_ic.max() / 2,
               label=f'r = {r:.2f}\nρ = {rho:.2f}', ha='left', va='top', size=12)
    + theme_cowplot()
    + labs(x='Effective_bp', y='pattern_0 core IC (effective bp)', color='density')
    + guides(color=guide_colorbar(display='rectangles', nbin=100))
    + theme(figure_size=(8, 8))
)
ic_comp_p0.show()
ic_comp_p0.save(filename=fig_path('ic_com_p0.pdf'))